# Historical cholera: a location is not a person

How does a map change when rows count locations but symbols represent recorded deaths? We compare a spot map, an aggregation and derived distances without claiming that spatial proximity proves a disease mechanism.

## Objectives

- Distinguish 250 mapped locations from 489 recorded deaths
- Use symbol area and weighted summaries correctly
- Separate historical records, geometric distances and causal evidence

## Start in a fresh kernel

Run every code cell from the top. The Book shows publication-time outputs;
JupyterLite uses the Python (Pyodide) kernel and downloads its runtime and
packages initially. All observations and metadata below come from packaged
snapshots: no live API, token, paid service or GPU is required. If a file is
missing or modified, validation fails explicitly; restore the supplied files.
Download notebook edits and exports to keep them outside browser storage.
The Python code is visible so that transformations can be inspected.

In [ ]:
import sys
from pathlib import Path

if sys.platform == "emscripten":
    import piplite
    await piplite.install(["numpy", "pandas", "matplotlib"])

# Book starts here; desktop/Lite may start at either the notebook or content root.
candidates = [Path.cwd(), Path.cwd() / "repository_examples",
              Path.cwd() / "content" / "repository_examples"]
lesson_dir = next((p for p in candidates if (p / "repo_examples.py").is_file()), None)
if lesson_dir is None:
    raise FileNotFoundError("Open this lesson from content/repository_examples on desktop, or repository_examples in Lite, with repo_examples.py and data/ alongside it.")
sys.path.insert(0, str(lesson_dir))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML
import repo_examples as rx

if sys.platform != "emscripten":
    get_ipython().run_line_magic("matplotlib", "inline")

plt.rcParams.update({"figure.figsize": (9, 4.8), "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 10})
display(rx.validate_all())

## Evidence and attribution

The cards describe the evidence type before any visualization. The manifest retains exact URLs/queries, retrieval UTC, SHA-256 checksums, field order, row counts and transformation recipes. Dataset terms remain separate from the undecided project license.

In [ ]:
display(rx.source_card("cholera"))

In [ ]:
deaths = rx.read_csv("cholera/cholera_deaths.csv")
pumps = rx.read_csv("cholera/pumps.csv")
assert len(deaths) == 250 and deaths.DEATHS.sum() == 489 and len(pumps) == 8
assert deaths.DEATHS.gt(0).all()
display(deaths.head())
display(pumps)
# A local equirectangular approximation, origin at the sample mean, in meters.
# Suitable for this small neighborhood; not a walking network or survey-grade CRS.
lon0, lat0 = deaths.LON.mean(), deaths.LAT.mean()
R = 6_371_000
dx = np.radians(deaths.LON.to_numpy()-lon0) * R * np.cos(np.radians(lat0))
dy = np.radians(deaths.LAT.to_numpy()-lat0) * R
px = np.radians(pumps.LON.to_numpy()-lon0) * R * np.cos(np.radians(lat0))
py = np.radians(pumps.LAT.to_numpy()-lat0) * R
weights = deaths.DEATHS.to_numpy()

## View 1: symbol area represents recorded deaths

Matplotlib's `s` is marker area in points squared. Multiplying it by DEATHS gives proportional areas; multiplying radius by DEATHS would exaggerate large values. The unweighted center gives every address equal influence, while the weighted center gives each recorded death equal influence. Neither identifies a source of infection.

**Method connection:** start with the catalog's Cartesian coordinates (§2), Table (§3) and Scatterplot (§9). Point maps and proportional symbols extend those constructions to geographic evidence; hexbin and empirical cumulative distribution views add aggregation and weighted-distribution questions. The retained GPL data notice is separate from the new original code and prose.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
ax.scatter(dx, dy, s=weights*12, alpha=.45, color="#a0473e", label="Area ∝ recorded deaths")
ax.scatter(px, py, marker="^", s=70, color="#1d6d8d", label="8 mapped pumps")
for i, row in pumps.iterrows():
    ax.annotate(str(row.FID), (px[i], py[i]), xytext=(4, 4), textcoords="offset points", fontsize=8)
ax.scatter(np.mean(dx), np.mean(dy), marker="x", s=100, color="black", label="Location-weighted center")
ax.scatter(np.average(dx, weights=weights), np.average(dy, weights=weights), marker="+", s=150, color="#e3a326", label="Death-weighted center")
ax.set_aspect("equal")
rx.finish(ax, "Historical digitization — no modern basemap", "Approximate east displacement (m)", "Approximate north displacement (m)")
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

## View 2: aggregation changes the unit

Hexagons sum the DEATHS field. They do not count unique persons independently verified by this course, estimate a population rate, or resolve geocoding error. Changing grid resolution changes the visual surface even though the recorded total remains 489.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)
for ax, resolution in zip(axes, [12, 24]):
    image = ax.hexbin(dx, dy, C=weights, reduce_C_function=np.sum, gridsize=resolution,
                      mincnt=1, cmap="YlOrRd", vmin=0, vmax=50)
    assert np.isclose(np.ma.sum(image.get_array()), weights.sum()), "A hexbin dropped source weight"
    ax.scatter(px, py, marker="^", color="#216b8b", s=25)
    ax.set_aspect("equal")
    rx.finish(ax, f"Grid resolution {resolution}; summed deaths", "East (m)", "North (m)")
fig.colorbar(image, ax=axes, label="Recorded deaths per occupied bin; fixed scale")
plt.show()

## View 3: derived distances have a different evidence status

We calculate straight-line distances from the approximate local coordinates to every mapped pump, then choose the nearest. The weighted empirical cumulative distribution expands each location's influence by its recorded deaths; it does not create person-level exposure observations. Street names are labels, not a road network. Distance does not identify which pump someone used.

In [ ]:
distances = np.hypot(dx[:, None]-px[None, :], dy[:, None]-py[None, :])
nearest = distances.argmin(axis=1)
nearest_m = distances.min(axis=1)
summary = pd.DataFrame({"pump_FID": pumps.FID, "Street": pumps.Street,
                        "nearest_locations": np.bincount(nearest, minlength=len(pumps)),
                        "associated_recorded_deaths": np.bincount(nearest, weights=weights, minlength=len(pumps))})
assert summary.associated_recorded_deaths.sum() == 489
display(summary)
order = np.argsort(nearest_m)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].step(nearest_m[order], np.cumsum(weights[order])/weights.sum(), where="post", label="Death-weighted")
axes[0].step(nearest_m[order], np.arange(1, len(order)+1)/len(order), where="post", label="Location-weighted")
rx.finish(axes[0], "Nearest mapped pump (derived)", "Approximate straight-line distance (m)", "Cumulative fraction")
axes[0].legend()
axes[1].bar(summary.pump_FID.astype(str), summary.associated_recorded_deaths, color="#a0473e")
rx.finish(axes[1], "Geometric nearest-pump assignment", "Pump FID", "Associated recorded deaths (not exposure)")
fig.tight_layout()
plt.show()

## Exercise

Change the hexagon resolution and the symbol-area multiplier. Which changes affect the recorded total, and which change only perception? Then compare the nearest-pump table with an unweighted table. Write one descriptive spatial claim and identify the denominators, dates and contextual evidence a causal claim would require.

## Limitations and reflection

Modern digitization completeness and original transcription are not independently verified. Historical locations contain multiple recorded deaths. There are no dates, population denominators, controls, consumption records or walking routes. The local distance approximation is useful for visualization, not exposure assessment. A spot map cannot establish disease causation.

## Citations and repository connections

Data and retained GPL-3.0 terms: [GeoLibre-GeoAI cholera bundle at dda3a4667a12](https://github.com/jltobias/JupyterLite-GeoLibre-GeoAI/tree/dda3a4667a1267b5ac4858723c46d8c0c82cb7a6/book/notebooks/data/cholera), identifying [PHI browser cholera at 476f6dd5a843](https://github.com/PHI-Case-Studies/Jupyterlite-1854-Cholera-Basic/tree/476f6dd5a8430e6eb823e79614fda1d7e12a78fb). Comparison reference: [PHI basic case at 3d02bafc779c](https://github.com/PHI-Case-Studies/1854-Cholera-Outbreak-London-Basic/tree/3d02bafc779c03d3195738b3d110e43a8184cfaf). No authored maps or figures were copied.

See the [companion index](index.md) and [full attribution](https://jltobias.github.io/JupyterLite-Data-Visualization/book/references.html). These are original teaching code and narrative; linked repository notebooks and authored figures have not been copied.